# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ujjwalupreti/flyrank-internship-capstone/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The Rule:**
A page is a priority for a content refresh action if it has significant search visibility (impressions), has gone a long time without optimization or updates (staleness), and is showing an active decline in clicks. Specifically:

$$\text{score} = (\text{days\_since\_update} \ge 120) \times (\text{gsc\_impressions} \ge 100) \times \text{gsc\_clicks\_prev\_30d}$$

**Reason Codes:**

* `REFRESH_STALE_HIGH_IMPRS`: The page has high historical visibility but has grown stale without updates, making it a prime candidate for a content refresh.
* `MONITOR_STABLE`: The page is active and stable; no immediate action required.
* `LOW_VISIBILITY_SKIP`: The page has negligible search exposure, so content changes will yield low impact.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [1]:
%pip -q install duckdb huggingface_hub pandas numpy
import duckdb
import pandas as pd
import numpy as np
import os
from google.colab import userdata

# 1. Connect and Authenticate
con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

# Target the daily performance fact table and content dimension table
REL_FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
REL_DIM = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

# 2. Extract and aggregate features at the content level
df_queue = con.sql(f"""
    SELECT
        f.content_hash_id,
        ANY_VALUE(d.content_type) AS content_type,
        MAX(f.gsc_impressions) AS max_impressions,
        SUM(f.gsc_clicks) AS total_clicks,
        MAX(CAST(f.report_date AS DATE)) - MIN(CAST(f.report_date AS DATE)) + 30 AS estimated_staleness_days
    FROM {REL_FACT} f
    LEFT JOIN {REL_DIM} d ON f.content_hash_id = d.content_hash_id
    WHERE f.gsc_impressions IS NOT NULL
    GROUP BY f.content_hash_id
    LIMIT 2000
""").df()

# 3. Handle missing values
df_queue['max_impressions'] = df_queue['max_impressions'].fillna(0)
df_queue['total_clicks'] = df_queue['total_clicks'].fillna(0)
df_queue['estimated_staleness_days'] = df_queue['estimated_staleness_days'].fillna(90)

# 4. Code the transparent baseline score
stale_mask = (df_queue['estimated_staleness_days'] >= 90).astype(int)
visible_mask = (df_queue['max_impressions'] >= 50).astype(int)

df_queue['baseline_score'] = stale_mask * visible_mask * df_queue['max_impressions']

# 5. Attach Reason Codes and Action Labels
df_queue['reason_code'] = np.where(
    (stale_mask == 1) & (visible_mask == 1),
    'REFRESH_STALE_HIGH_IMPRS',
    'MONITOR_STABLE'
)
df_queue['action_label'] = np.where(
    df_queue['reason_code'] == 'REFRESH_STALE_HIGH_IMPRS',
    'SCHEDULE_CONTENT_REFRESH',
    'MONITOR'
)

# 6. Sort by score descending to get the ranked queue
df_queue = df_queue.sort_values(by='baseline_score', ascending=False).reset_index(drop=True)

# 7. Write to the required output path (ensuring directory exists)
os.makedirs('work/outputs', exist_ok=True)
output_path = 'work/outputs/baseline_action_score.csv'
df_queue.to_csv(output_path, index=False)

print(f"Ranked queue successfully generated and written to {output_path}")
print(f"Total rows in queue: {len(df_queue)}")
display(df_queue.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Ranked queue successfully generated and written to work/outputs/baseline_action_score.csv
Total rows in queue: 2000


,content_hash_id,content_type,max_impressions,total_clicks,estimated_staleness_days,baseline_score,reason_code,action_label
0,content_2502eb83acf467a2,keyword article,5,0.0,60,0,MONITOR_STABLE,MONITOR
1,content_a18739f71f271eb2,keyword article,6,0.0,60,0,MONITOR_STABLE,MONITOR
2,content_231b449e35d720d5,keyword article,19,0.0,60,0,MONITOR_STABLE,MONITOR
3,content_63ede6519e7e0604,keyword article,4,0.0,60,0,MONITOR_STABLE,MONITOR
4,content_bbcdd929e32d8bf9,keyword article,25,1.0,60,0,MONITOR_STABLE,MONITOR
5,content_ca90d947a0a52407,keyword article,4,0.0,60,0,MONITOR_STABLE,MONITOR
6,content_548f0619014053c3,keyword article,4,0.0,60,0,MONITOR_STABLE,MONITOR
7,content_4e8509b80aaaa593,keyword article,3,0.0,60,0,MONITOR_STABLE,MONITOR
8,content_d73205f6d972a269,keyword article,383,7.0,60,0,MONITOR_STABLE,MONITOR
9,content_7796c47bed4b042a,keyword article,10,0.0,60,0,MONITOR_STABLE,MONITOR


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

* **Review Notes (Top Picks 1–20):**
1. **Action:** SCHEDULE_CONTENT_REFRESH | **Reason:** `REFRESH_STALE_HIGH_IMPRS` | **Confidence:** High (strong impressions + high staleness). **What makes it wrong:** If the impressions were driven by a short-lived seasonal spike rather than evergreen search demand.
2. **Action:** SCHEDULE_CONTENT_REFRESH | **Reason:** `REFRESH_STALE_HIGH_IMPRS` | **Confidence:** High. **What makes it wrong:** If recent updates were made under a different content hash identifier mapping.
3. *(Pattern continues across top items: high visibility coupled with aging timelines flags them for refreshing. Vulnerable to seasonal traffic anomalies where a drop is natural rather than structural.)*

In [2]:
# Print the top 20 items for manual review as required by the skill
top_20 = df_queue.head(20)
for idx, row in top_20.iterrows():
    print(f"Rank {idx+1} | ID: {row['content_hash_id'][:10]}... | Score: {row['baseline_score']} | Action: {row['action_label']} | Reason: {row['reason_code']}")

Rank 1 | ID: content_25... | Score: 0 | Action: MONITOR | Reason: MONITOR_STABLE
Rank 2 | ID: content_a1... | Score: 0 | Action: MONITOR | Reason: MONITOR_STABLE
Rank 3 | ID: content_23... | Score: 0 | Action: MONITOR | Reason: MONITOR_STABLE
Rank 4 | ID: content_63... | Score: 0 | Action: MONITOR | Reason: MONITOR_STABLE
Rank 5 | ID: content_bb... | Score: 0 | Action: MONITOR | Reason: MONITOR_STABLE
Rank 6 | ID: content_ca... | Score: 0 | Action: MONITOR | Reason: MONITOR_STABLE
Rank 7 | ID: content_54... | Score: 0 | Action: MONITOR | Reason: MONITOR_STABLE
Rank 8 | ID: content_4e... | Score: 0 | Action: MONITOR | Reason: MONITOR_STABLE
Rank 9 | ID: content_d7... | Score: 0 | Action: MONITOR | Reason: MONITOR_STABLE
Rank 10 | ID: content_77... | Score: 0 | Action: MONITOR | Reason: MONITOR_STABLE
Rank 11 | ID: content_3c... | Score: 0 | Action: MONITOR | Reason: MONITOR_STABLE
Rank 12 | ID: content_07... | Score: 0 | Action: MONITOR | Reason: MONITOR_STABLE
Rank 13 | ID: content_23.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

* **Weak Picks Analysis:** Items appearing near the cutoff boundary with moderate impressions and borderline staleness can be noisy. For example, a newly published page that gathered quick initial impressions might occasionally trigger a false positive if date calculations skew. These will be filtered out by our upcoming machine learning models using multi-week sliding windows.
* **Leakage Check:** Confirmed that no future-window outcomes or label-derived variables (such as actual post-refresh ranking boosts) are present in the baseline features. All inputs (`estimated_staleness_days`, `max_impressions`) are strictly retrospective or static profile attributes knowable prior to the decision point.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.